# ⚽ Football Vision — Analisi Tattica su GPU (Colab)

Questo notebook fa **tutto su GPU gratuita**: scarica il codice e i modelli, prende una clip
da YouTube, la analizza e ti mostra **radar 2D + heatmap + dashboard + statistiche**.

Sul tuo PC il modello del campo va a ~4 s/frame; **qui su GPU va a ~0,1 s/frame** (40× più veloce).

## Come si usa (5 passi)
1. In alto: menu **Runtime → Cambia tipo di runtime → GPU (T4)** → Salva.
2. Esegui le celle **in ordine** (clic sulla cella + `Shift+Invio`).
3. Nella cella 'Scegli la clip' incolla un **link YouTube** e l'intervallo.
4. Esegui l'analisi.
5. Guarda i risultati e scaricali.

## 1) Verifica GPU

In [ ]:
import torch
if torch.cuda.is_available():
    print('✅ GPU attiva:', torch.cuda.get_device_name(0))
else:
    print('❌ GPU NON attiva! Vai su Runtime → Cambia tipo di runtime → GPU (T4) e riavvia.')

## 2) Scarica codice + modelli + librerie (~1-2 min, una volta sola)

In [ ]:
# Librerie
!pip -q install ultralytics supervision scikit-learn yt-dlp 2>/dev/null

# Codice dal repository GitHub (sempre aggiornato)
import os, shutil
if os.path.exists('football-vision'):
    shutil.rmtree('football-vision')
!git clone -q https://github.com/sebavidal2001/football-vision.git
%cd football-vision

# Modelli AI da Hugging Face (campo + giocatori)
import urllib.request
os.makedirs('vista_tattica', exist_ok=True)
modelli = {
    'vista_tattica/yolo-football-pitch-detection.pt':
        'https://huggingface.co/martinjolif/yolo-football-pitch-detection/resolve/main/yolo-football-pitch-detection.pt',
    'vista_tattica/giocatori_calcio.pt':
        'https://huggingface.co/uisikdag/yolo-v8-football-players-detection/resolve/main/best.pt',
}
for dst, url in modelli.items():
    if not os.path.exists(dst):
        print('Scarico', os.path.basename(dst), '...')
        urllib.request.urlretrieve(url, dst)
print('\n✅ Tutto pronto.')

## 3) Scegli la clip da analizzare
Incolla un **link YouTube** (meglio una *tactical cam / panoramic*) e l'intervallo da ritagliare.
Tieni la clip entro **1-3 minuti** per la prima prova.

In [ ]:
LINK   = 'https://youtu.be/gzNLfgbxsLk'   # <-- il tuo link YouTube
INIZIO = '00:10:00'                        # <-- inizio (h:mm:ss)
FINE   = '00:11:40'                        # <-- fine   (h:mm:ss)

import os
os.makedirs('clips_input', exist_ok=True)
out = 'clips_input/clip_input.mp4'
if os.path.exists(out):
    os.remove(out)
!yt-dlp --download-sections "*{INIZIO}-{FINE}" \
  -f "bestvideo[height<=720]+bestaudio/best[height<=720]" \
  --merge-output-format mp4 -o "{out}" "{LINK}"
print('\n✅ Clip pronta:', out if os.path.exists(out) else 'ERRORE')

*In alternativa,* per **caricare un file dal tuo PC** invece di YouTube, scommenta ed esegui:

In [ ]:
# from google.colab import files
# os.makedirs('clips_input', exist_ok=True)
# up = files.upload()
# import shutil
# src = list(up.keys())[0]
# shutil.move(src, 'clips_input/clip_input.mp4')
# print('Caricata.')

## 4) Analisi su GPU (radar + heatmap + statistiche + dashboard)
Su GPU è veloce: puoi tenere `SALTO=2` e `OGNI_CAMPO=1` (qualità alta).

In [ ]:
SALTO = 2          # analizza 1 frame ogni N (su GPU puoi tenerlo basso)
OGNI_CAMPO = 1     # riconosci il campo ogni N frame

import glob
video = 'clips_input/clip_input.mp4'
base = os.path.splitext(os.path.basename(video))[0]

print('▶ Radar 2D + rilevamento giocatori...')
!python vista_tattica/genera_radar_auto.py "{video}" --salto {SALTO} --ogni_campo {OGNI_CAMPO} --imgsz 1280

csv_pos = f'output/POSIZIONIAUTO_{base}.csv'
print('\n▶ Heatmap + statistiche per giocatore...')
!python analisi/stats_giocatori.py "{csv_pos}" --min_rilevazioni 20

print('\n▶ Dashboard di confronto...')
!python analisi/confronto_giocatori.py "output/STATISTICHE_{base}.csv"
print('\n✅ FATTO.')

## 5) Guarda i risultati

In [ ]:
from IPython.display import Image, Video, display
import glob, os

# Dashboard di confronto
dash = glob.glob('output/DASHBOARD_*.png')
if dash:
    print('=== DASHBOARD DI CONFRONTO ===')
    display(Image(dash[0]))

# Heatmap di squadra
for f in sorted(glob.glob('output/heatmaps_*/_SQUADRA_*.png')):
    display(Image(f, width=520))

In [ ]:
# Video radar (ricodificato per il browser)
import glob
rad = glob.glob('output/RADARAUTO_*.mp4')
if rad:
    !ffmpeg -y -i "{rad[0]}" -vcodec libx264 -pix_fmt yuv420p output/_radar_web.mp4 2>/dev/null
    from IPython.display import Video, display
    display(Video('output/_radar_web.mp4', embed=True, width=900))

## 6) Scarica tutti i risultati (zip)

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('risultati', 'zip', 'output')
files.download('risultati.zip')